In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df=pd.read_csv("train.csv")

In [ ]:
df

In [ ]:
df=df.drop(columns=['id','keyword','location'])

In [ ]:
df

In [ ]:
df.shape

In [ ]:
df['target'].value_counts()

In [ ]:
df.info()

#### No null values

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X=df['text']
y=df['target']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
y_test

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

In [ ]:
train_df=pd.DataFrame({'text':X_train,'target':y_train})
val_df=pd.DataFrame({'text':X_test,'target':y_test})

In [ ]:
train_dataset=Dataset.from_pandas(train_df,preserve_index=False)
val_dataset=Dataset.from_pandas(val_df,preserve_index=False)

In [ ]:
checkpoint="distilbert-base-uncased"

In [ ]:
tokenizer=AutoTokenizer.from_pretrained(checkpoint)

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
# padding= max_length as when we take a batch of 1000 with padding=true the length of max sequence will be different for different batches that is why padding = "max_length"
def tokenize_function(example):
  return tokenizer(example["text"], padding='max_length',truncation=True , max_length=128)

In [ ]:
tokenized_train=train_dataset.map(tokenize_function,batched=True)
tokenized_val=val_dataset.map(tokenize_function,batched=True)

In [ ]:
tokenized_train

In [ ]:
tokenized_train=tokenized_train.rename_column("target","labels")
tokenized_val=tokenized_val.rename_column("target","labels")
tokenized_train=tokenized_train.remove_columns("text")
tokenized_val=tokenized_val.remove_columns("text")

In [ ]:
tokenized_train

In [ ]:
from transformers import AutoModelForSequenceClassification

In [ ]:
model=AutoModelForSequenceClassification.from_pretrained(checkpoint,num_labels=2)

In [ ]:
from transformers import TrainingArguments,Trainer

In [ ]:
from sklearn.metrics import accuracy_score,f1_score

In [ ]:
training_args=TrainingArguments("test-trainer",eval_strategy="epoch",num_train_epochs=3,per_device_train_batch_size=16, per_device_eval_batch_size=16, save_strategy="epoch", load_best_model_at_end=True)


In [ ]:
def compute_metrics(eval_preds):
  logits,labels=eval_preds
  predictions = np.argmax(logits, axis=-1)
  return {
      "accuracy": accuracy_score(labels,predictions),
      "f1": f1_score(labels,predictions)
  }

In [ ]:
trainer=Trainer(
    model,
    training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
trainer.save_model("./saved_model")
tokenizer.save_pretrained("./saved_model")
print("Model Saved")

In [ ]:
from transformers import pipeline

In [ ]:
classifier=pipeline("text-classification",model="./saved_model")

In [ ]:
test_tweets = [
    "Forest fire near La Ronge Sask. Canada",
    "I love sunny days at the beach",
    "Earthquake hits California, thousands evacuated",
    "Just had the best pizza of my life"
]

In [ ]:
for tweet in test_tweets:
  result=classifier(tweet)[0]
  label="Disaster" if result["label"]=="LABEL_1" else "Not Disaster"
  print(f"Tweet: {tweet}")
  print(f"Label : {label}")

In [ ]:
import shutil
shutil.make_archive("saved_model", "zip", "../backend/saved_model")